# 🌊 Flood Depth Model - Google Colab GPU Training
## Run Model Improvements on FREE GPU in Minutes!

**What you'll do:**
1. Enable GPU (free, automatic)
2. Clone your model from GitHub
3. Upload training data
4. Run improvement strategy (choose one: 0 min to 3 hours)
5. Download improved model

**Cost:** $0 (completely free!)  
**Time:** Choose your path - instant to 3 hours

## STEP 1: Enable GPU
⚠️ IMPORTANT: Must do this first!

1. Click **Runtime** (top menu)
2. Click **Change runtime type**
3. Select **GPU** from Hardware accelerator dropdown
4. Click **Save**

Then run this cell to verify:

In [ ]:
# Verify GPU is available
import torch
print(f"🔍 GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ GPU not detected! Go to Runtime → Change runtime type → Select GPU")

# Show GPU info
!nvidia-smi

## STEP 2: Mount Google Drive
Lets you save files and access Drive folders

In [ ]:
from google.colab import drive

# Mount drive (will ask for permission)
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Show drive contents
import os
!ls /content/drive/My\ Drive | head -20

## STEP 3: Clone GitHub Repository

In [ ]:
import os

# Change to Drive directory
os.chdir('/content/drive/My Drive')

# Clone repo (only if not already cloned)
if not os.path.exists('flood-depth-estimator'):
    !git clone https://github.com/Mishra-Kaumod/flood-depth-estimator.git
    print("✅ Repository cloned!")
else:
    print("✅ Repository already exists")

# Change to repo directory
os.chdir('flood-depth-estimator')
print(f"✅ Working directory: {os.getcwd()}")

# List main files
print("\n📁 Repository contents:")
!ls -la | grep -E '\.(py|md|yaml)$' | head -15

## STEP 4: Install Dependencies

In [ ]:
# Install PyTorch with CUDA support
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install -q pillow opencv-python numpy tqdm scikit-learn

print("✅ All dependencies installed!")

## STEP 5: Upload Training Data
Choose Option A or B:

### Option A: Upload Files via Browser (Best for small data <100MB)

In [ ]:
from google.colab import files
import os
import shutil

# Create data directories
os.makedirs('data/train/images', exist_ok=True)
os.makedirs('data/val/images', exist_ok=True)

print("📤 Click 'Choose Files' to upload your images")
print("   Select training images (jpg, png, jpeg)")
print("   Can upload multiple at once!\n")

uploaded = files.upload()

# Move uploaded files to data folder
for filename in uploaded.keys():
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        # Check if it's for train or val
        if 'val' in filename.lower():
            dst = f'data/val/images/{filename}'
        else:
            dst = f'data/train/images/{filename}'
        
        shutil.move(filename, dst)
        print(f"✅ Moved: {filename} → {dst}")

# Show what we have
train_count = len(os.listdir('data/train/images'))
val_count = len(os.listdir('data/val/images'))

print(f"\n✅ Data ready!")
print(f"   Training images: {train_count}")
print(f"   Validation images: {val_count}")

### Option B: Use Existing Data from Google Drive (Better for large data)

In [ ]:
import os
import shutil

# Create data directories
os.makedirs('data/train/images', exist_ok=True)
os.makedirs('data/val/images', exist_ok=True)

# Edit these paths to match your Google Drive structure
drive_train = '/content/drive/My Drive/your_flood_data/train'  # ← CHANGE THIS
drive_val = '/content/drive/My Drive/your_flood_data/val'      # ← CHANGE THIS

# Copy from Drive
if os.path.exists(drive_train):
    os.system(f'cp -r {drive_train}/* data/train/images/')
    print("✅ Training data copied from Drive")

if os.path.exists(drive_val):
    os.system(f'cp -r {drive_val}/* data/val/images/')
    print("✅ Validation data copied from Drive")

# Show what we have
train_count = len(os.listdir('data/train/images'))
val_count = len(os.listdir('data/val/images'))

print(f"\n✅ Data ready!")
print(f"   Training images: {train_count}")
print(f"   Validation images: {val_count}")

## STEP 6: Choose Your Improvement Strategy
### ⚡ Pick ONE below and run it

### 🔥 OPTION 1: TEST-TIME AUGMENTATION (INSTANT - 0 MINUTES!)
**Best if:** You need results right now, no training  
**Expected:** -3-8% improvement  
**GPU needed:** NO

In [ ]:
import os
os.chdir('/content/drive/My Drive/flood-depth-estimator')

# Run test-time augmentation
!python test_time_augmentation.py \
    --model models/best_flood_model.pth \
    --image data/val/images/test_image.jpg \
    --num-augs 7

print("\n✅ Done! Check results:")
print("   • Result file: test_image_tta_results.json")
print("   • Use result['mean'] for final prediction")

### 🎯 OPTION 2: LIGHTWEIGHT FINE-TUNING (30 MINUTES)
**Best if:** You have 30 minutes and want quick improvement  
**Expected:** -5-15% improvement  
**GPU needed:** Optional (works on CPU too, but GPU faster)

In [ ]:
import os
os.chdir('/content/drive/My Drive/flood-depth-estimator')

# Run lightweight fine-tuning
!python fine_tune_head.py \
    --checkpoint models/best_flood_model.pth \
    --train-dir data/train/images \
    --val-dir data/val/images \
    --epochs 5 \
    --batch-size 32 \
    --device cuda

print("\n✅ Fine-tuning complete!")
print("   • New model: models/best_flood_model_finetuned.pth")
print("   • Training history: best_flood_model_finetuned_history.json")

### 💎 OPTION 3: WATER-AWARE FINE-TUNING (2-3 HOURS - BEST!)
**Best if:** You have 2-3 hours and want best non-retrain improvement  
**Expected:** -20-40% improvement  
**GPU needed:** YES (Colab T4 is sufficient)

⚠️ **Note:** This takes 2-3 hours. Keep Colab open during training!

In [ ]:
import os
os.chdir('/content/drive/My Drive/flood-depth-estimator')

# First, check if script exists
if not os.path.exists('fine_tune_water_aware.py'):
    print("❌ fine_tune_water_aware.py not found")
    print("ℹ️  Using src/train_water_aware.py instead...")
    print("Run full retraining (4-6 hours):")
    print()
    !python src/train_water_aware.py --config config/config.yaml
else:
    # Run water-aware fine-tuning
    !python fine_tune_water_aware.py \
        --checkpoint models/best_flood_model.pth \
        --train-dir data/train/images \
        --val-dir data/val/images \
        --epochs 5 \
        --batch-size 32 \
        --device cuda

print("\n✅ Water-aware fine-tuning complete!")
print("   • New model: models/best_flood_model_water_aware.pth")

## STEP 7: Check Results

In [ ]:
import os
import json
os.chdir('/content/drive/My Drive/flood-depth-estimator')

# List model files
print("📦 Available models:")
models = os.listdir('models')
for model in sorted(models):
    path = f'models/{model}'
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"   ✓ {model:<40} ({size_mb:.1f} MB)")

# Check for training results
print("\n📊 Training results:")
results = [f for f in os.listdir('.') if 'history' in f or 'results' in f]
for result in sorted(results):
    print(f"   ✓ {result}")
    if result.endswith('.json'):
        with open(result, 'r') as f:
            data = json.load(f)
            print(f"      Keys: {list(data.keys())}")

## STEP 8: Download Results

### Option A: Save to Google Drive (Recommended)

In [ ]:
import os
import shutil

os.chdir('/content/drive/My Drive/flood-depth-estimator')

# Copy models to Google Drive for easy access
dest_dir = '/content/drive/My Drive/flood_models_results'
os.makedirs(dest_dir, exist_ok=True)

# Copy all models
for model in os.listdir('models'):
    src = f'models/{model}'
    if os.path.isfile(src):
        dst = f'{dest_dir}/{model}'
        shutil.copy(src, dst)
        print(f"✅ Copied: {model}")

print(f"\n✅ All models saved to Google Drive!")
print(f"   📁 Location: {dest_dir}")
print(f"   ✓ You can access them anytime from your Drive")

### Option B: Download to Computer

In [ ]:
from google.colab import files
import os

os.chdir('/content/drive/My Drive/flood-depth-estimator')

# Download the improved model
model_to_download = 'models/best_flood_model_finetuned.pth'  # ← Change if using different model

if os.path.exists(model_to_download):
    print(f"📥 Downloading: {model_to_download}")
    files.download(model_to_download)
    print("✅ Download started!")
else:
    print(f"❌ File not found: {model_to_download}")
    print(f"\nAvailable models:")
    for model in os.listdir('models'):
        print(f"  • models/{model}")

## 🎉 You're Done!

### Summary:
✅ GPU enabled on Colab  
✅ Repository cloned from GitHub  
✅ Training data uploaded  
✅ Model improved using your chosen strategy  
✅ Results saved/downloaded  

### Next Steps:
1. Download your improved model
2. Use it in your local environment
3. Test on your data
4. Deploy if satisfied!

**All at ZERO cost on free Colab GPU!** 🚀

---

## 📞 Troubleshooting

### GPU not showing?
- Go to **Runtime** → **Change runtime type** → Select **GPU** → **Save**
- Re-run the GPU verification cell

### CUDA out of memory?
- Reduce batch size: `--batch-size 16` (instead of 32)
- Or try: `--batch-size 8`

### Data not found?
- Check file paths in upload cells
- Make sure images are in `.jpg`, `.png`, or `.jpeg` format
- Run: `!ls data/train/images` to verify

### Training too slow?
- Reduce number of epochs: `--epochs 3` (instead of 5)
- Check GPU is actually being used: `!nvidia-smi`

### Disconnected during training?
- Keep Colab tab open while training
- Training stops if browser closes
- Results saved to Google Drive automatically